## XGBoost on MIMIC IV data

First let us load the data (already preprocessed on tabular_preprocessing.ipynb).

In [ ]:
# Importing libraries
import ast
import xgboost as xgb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.sparse as sp


from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from scipy.sparse import hstack


In [14]:
# Load dataset
df = pd.read_csv("mimiciv_clinical_dataset_tabular_death_visit.csv")

# Count admission categories
admission_counts = df['admission_category'].value_counts()

# Display counts
print("Admission category counts:")
print(admission_counts := df['admission_category'].value_counts())

Admission category counts:
NORMAL       291667
EMERGENCY    254361
Name: admission_category, dtype: int64


In [15]:
df.head(3)

,subject_id,hadm_id,age_at_event,gender,admission_type,admission_category,discharge_location,death_flag,age_at_death,died_during_visit,days_from_last_visit_to_death,days_until_next_visit,icu_admissions,icu_days,num_diagnoses,diagnosis_list,num_procedures,procedure_list,num_medications,medication_list
0,10000032,22595853,52,F,URGENT,EMERGENCY,HOME,1,52.0,0,32.0,50.0,0.0,0.0,8.0,"['Portal hypertension', 'Other ascites', 'Cirr...",1.0,['Percutaneous abdominal drainage'],14.0,"['Furosemide', 'Ipratropium Bromide Neb', 'Pot..."
1,10000032,22841357,52,F,EW EMER.,EMERGENCY,HOME,1,52.0,0,32.0,25.0,0.0,0.0,8.0,['Unspecified viral hepatitis C with hepatic c...,1.0,['Percutaneous abdominal drainage'],15.0,"['Furosemide', 'Rifaximin', 'Sodium Chloride 0..."
2,10000032,29079034,52,F,EW EMER.,EMERGENCY,HOME,1,52.0,0,32.0,11.0,1.0,0.0,13.0,"['Other iatrogenic hypotension', 'Chronic hepa...",0.0,None,24.0,"['Bisacodyl', 'Senna', 'Calcium Carbonate', 'R..."


In [21]:
patients = df['subject_id'].unique()
train_patients, test_patients = train_test_split(patients, test_size=0.2, random_state=42)

train_df = df[df['subject_id'].isin(train_patients)].copy()
test_df = df[df['subject_id'].isin(test_patients)].copy()

In [25]:
# Be sure medication_list is a list, e.g.: ['Furosemide', 'Bisacodyl', ...]

def safe_convert(x):
    if pd.isna(x):
        return ''
    elif isinstance(x, list):
        return ' '.join(x)
    elif isinstance(x, str):
        try:
            x_eval = ast.literal_eval(x)
            if isinstance(x_eval, list):
                return ' '.join(x_eval)
            else:
                return ''
        except:
            return ''
    else:
        return ''

# Apply safely to each column:
train_df['med_text'] = train_df['medication_list'].apply(safe_convert)
train_df['diag_text'] = train_df['diagnosis_list'].apply(safe_convert)
train_df['proc_text'] = train_df['procedure_list'].apply(safe_convert)

test_df['med_text'] = test_df['medication_list'].apply(safe_convert)
test_df['diag_text'] = test_df['diagnosis_list'].apply(safe_convert)
test_df['proc_text'] = test_df['procedure_list'].apply(safe_convert)

In [26]:
# FIT CountVectorizer ONLY on training data (ClinicalBench style):
# Define CountVectorizer clearly, limit max features if needed:
med_vectorizer = CountVectorizer(max_features=1000, min_df=10)
diag_vectorizer = CountVectorizer(max_features=1000, min_df=10)
proc_vectorizer = CountVectorizer(max_features=1000, min_df=10)

# Fit on training only:
train_med = med_vectorizer.fit_transform(train_df['med_text'])
train_diag = diag_vectorizer.fit_transform(train_df['diag_text'])
train_proc = proc_vectorizer.fit_transform(train_df['proc_text'])

# Transform test data using fitted training vocabulary:
test_med = med_vectorizer.transform(test_df['med_text'])
test_diag = diag_vectorizer.transform(test_df['diag_text'])
test_proc = proc_vectorizer.transform(test_df['proc_text'])

In [28]:
# Combine sparse matrices horizontally (efficient):
X_train_sparse = sp.hstack([train_med, train_diag, train_proc])
X_test_sparse = sp.hstack([test_med, test_diag, test_proc])

# Convert sparse matrices to arrays if required (optional):
X_train = X_train_sparse.tocsr()
X_test = X_test_sparse.tocsr()

In [ ]:
numeric_features = ['age_at_event', 'icu_admissions', 'icu_days', 'num_diagnoses', 'num_procedures', 'num_medications']
train_numeric = train_df[numeric_features].values
test_numeric = test_df[numeric_features].values

# Final combined features:
X_train_final = hstack([X_train, train_numeric])
X_test_final = hstack([X_test, test_numeric])

In [33]:
# Defining target variable
y_train = (train_df['days_from_last_visit_to_death'] <= 90).astype(int).values
y_test = (test_df['days_from_last_visit_to_death'] <= 90).astype(int).values

In [ ]:
xgb_model = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb_model.fit(X_train_final, y_train)

y_pred_proba = xgb_model.predict_proba(X_test_final)[:, 1]
y_pred = xgb_model.predict(X_test_final)

print(classification_report(y_test, y_pred))
print(f"AUC: {roc_auc_score(y_test, y_pred_proba):.3f}")

/root/miniforge3/envs/MIMICIV/lib/python3.8/site-packages/xgboost/core.py:158: UserWarning: [18:06:59] WARNING: /home/conda/feedstock_root/build_artifacts/xgboost-split_1727231492252/work/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


              precision    recall  f1-score   support

           0       0.87      0.96      0.91     87208
           1       0.71      0.43      0.53     22076

    accuracy                           0.85    109284
   macro avg       0.79      0.69      0.72    109284
weighted avg       0.84      0.85      0.83    109284

AUC: 0.872
